In [ ]:
import pandas as pd

columns = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent',
    'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count',
    'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

DATA_PATH = "s3://cmpe281-risk-detection/cmpe_281_data_models/data/raw/KDDTrain+.csv"
df = pd.read_csv(DATA_PATH, header=None, names=columns)

# 2. Drop metadata
df.drop(columns=['difficulty_level'], inplace=True)

# 3. Binary target
df['target'] = df['label'].apply(lambda x: 0 if x == 'normal' else 1)
df.drop(columns=['label'], inplace=True)

# 4. Encode categoricals - converting categorical text columns into machine-readable numeric form.

# One-hot encoding
df = pd.get_dummies(df, columns=['protocol_type', 'service', 'flag'])

# 5. Separate features and target
X = df.drop(columns=['target'])
y = df['target']

print("Shape:", df.shape)
print("Object cols left:", list(X.select_dtypes(include='object').columns))
print("Target balance:\n", y.value_counts())
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print(X.shape)
print(y.value_counts(normalize=True) * 100)

# Train Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# step 1 - Split the data into train and test

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
# Step 2: Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Step 3: Create Logistic Regression model
lr_model = LogisticRegression(max_iter=2000, random_state=42)

In [ ]:
# Step 4: Train and Predict on test data
lr_model.fit(X_train_scaled, y_train)

# Predict
y_pred = lr_model.predict(X_test_scaled)

In [ ]:
# Evaluate
print("Train Accuracy:", lr_model.score(X_train_scaled, y_train))
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))